# Jour 3 — Template de pipeline pour la compétition du jour

**Jour 3 — chapitre 02 : Mise en pratique**

## Comment utiliser ce notebook

Ce notebook n'est pas un TP avec une réponse unique : c'est un **squelette de pipeline**
réutilisable, que chaque participant adapte à la compétition Kaggle réellement choisie ce
jour-là (voir `speech/ressources-techniques.md` pour les options : *House Prices* en
valeur sûre, ou l'épisode actif de *Playground Series*).

Il est démontré ici sur le dataset Titanic, déjà familier, pour que la structure du
pipeline reste le point d'attention plutôt que la découverte d'un nouveau dataset.

Les cinq étapes suivent exactement l'énoncé de la diapositive 9 : lire l'énoncé, explorer
les données, construire une baseline, itérer, soumettre.


## Étape 1 — Lire l'énoncé : comprendre la target et la métrique de classement

**Question de réflexion :** avant d'écrire la moindre ligne de code sur la compétition
choisie, répondez par écrit à ces trois questions :

1. Quelle est exactement la colonne target, et sa nature (catégorielle ou continue) ?
2. Quelle métrique de classement est utilisée sur le leaderboard (accuracy, RMSE, AUC...) ?
3. Cette métrique privilégie-t-elle un type d'erreur particulier (comme vu en jour 2 pour
   precision/recall) ?


In [1]:
# Demonstration sur Titanic (a remplacer par le chargement de la vraie competition)

import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
})[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

titanic = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True)
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

features = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "Sex_male"]
features = [c for c in features if c in titanic.columns]

X = titanic[features]
y = titanic["Survived"]

titanic.head()


print("Target :", "Survived")
print("Nature de la target : catégorielle binaire -> classification")
print("Métrique typique pour ce type de probleme : accuracy, ou AUC si les classes sont desequilibrees")


Target : Survived
Nature de la target : catégorielle binaire -> classification
Métrique typique pour ce type de probleme : accuracy, ou AUC si les classes sont desequilibrees


## Étape 2 — Explorer les données : structure, valeurs manquantes, distributions

**Question de réflexion :** quelles étapes d'exploration, vues en jour 1 et 2, sont
systématiques quel que soit le dataset ?


In [2]:
titanic.info()
print()
titanic.isnull().sum()


<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Survived    891 non-null    int64  
 1   Pclass      891 non-null    int64  
 2   Age         891 non-null    float64
 3   SibSp       891 non-null    int64  
 4   Parch       891 non-null    int64  
 5   Fare        891 non-null    float64
 6   Sex_male    891 non-null    bool   
 7   Embarked_Q  891 non-null    bool   
 8   Embarked_S  891 non-null    bool   
 9   FamilySize  891 non-null    int64  
 10  IsAlone     891 non-null    int64  
dtypes: bool(3), float64(2), int64(6)
memory usage: 58.4 KB



Survived      0
Pclass        0
Age           0
SibSp         0
Parch         0
Fare          0
Sex_male      0
Embarked_Q    0
Embarked_S    0
FamilySize    0
IsAlone       0
dtype: int64

**Ce qu'on observe** : `.info()` et `.isnull().sum()` sont les deux premiers réflexes,
quel que soit le dataset de la compétition — ils orientent immédiatement vers les
décisions de nettoyage à prendre.


## Étape 3 — Construire une baseline : un premier modèle simple et rapide

**Question de réflexion :** pourquoi commencer volontairement par le modèle le plus
simple possible, plutôt que par l'algorithme le plus sophistiqué du programme ?


In [3]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Suivi des expériences avec MLflow

C'est précisément l'outil qui répond au point 5 de la check-list de la diapositive 12 (« a-t-on conservé une trace des expériences menées ? ») : plutôt qu'un tableur mis à jour à la main, chaque itération de cette section devient un run MLflow comparable aux autres.

Plutôt que de garder une trace manuelle des scores dans un carnet ou un tableur (voir la
check-list du jour 3 : « a-t-on conservé une trace des expériences menées ? »), on utilise
ici **MLflow** pour enregistrer automatiquement les paramètres et les métriques de chaque
essai.

Ceci suppose qu'un serveur de tracking MLflow tourne en local, lancé au préalable avec :

```bash
mlflow server --host 127.0.0.1 --port 5001 \
    --backend-store-uri sqlite:///mlflow_data/mlflow.db \
    --default-artifact-root ./mlflow_data/mlartifacts
```

L'interface est ensuite consultable dans un navigateur à l'adresse
http://127.0.0.1:5001. Voir `demos/README.md` pour le détail, et
`speech/ressources-techniques.md` pour son rôle dans la stack technique.

Si aucun serveur n'est joignable (par exemple sur Kaggle Notebooks ou Colab, qui ne
voient pas votre machine locale), remplacez la ligne `mlflow.set_tracking_uri(...)`
ci-dessous par `mlflow.set_tracking_uri("file:./mlruns")` : MLflow écrit alors ses runs
dans un simple dossier local, consultable plus tard avec `mlflow ui`.


In [4]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("OFDS - Jour 3 - Compétition du jour")


2026/09/13 17:24:21 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/mouaad/Formation/plb/PFDS/.venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/09/13 17:24:21 INFO mlflow.tracking.fluent: Experiment with name 'OFDS - Jour 3 - Compétition du jour' does not exist. Creating a new experiment.


<Experiment: artifact_location='/private/tmp/claude-501/-Users-mouaad-Formation-plb-PFDS/1618c5d4-eac3-4b9b-8c92-386b5ee7255b/scratchpad/mlflow_validation/mlartifacts/4', creation_time=1789313061644, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1789313061644, lifecycle_stage='active', name='OFDS - Jour 3 - Compétition du jour', tags={}, trace_location=None, workspace='default'>

In [5]:
baseline = LogisticRegression(max_iter=5000)

with mlflow.start_run(run_name="baseline"):
    baseline_scores = cross_val_score(baseline, X_train, y_train, cv=5)
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_metric("cv_accuracy_mean", baseline_scores.mean())

print("Score baseline (régression logistique) :", round(baseline_scores.mean(), 3))


View run baseline at: http://127.0.0.1:5001/#/experiments/4/runs/4476dd2ef6484cfa891fb3d6044e41c4
View experiment at: http://127.0.0.1:5001/#/experiments/4
Score baseline (régression logistique) : 0.803


**Ce qu'on observe / à retenir** : ce score devient la référence de la diapositive 17 du
jour 3 — « pourquoi construire une baseline ? pour disposer d'un point de comparaison
fiable et mesurer objectivement chaque amélioration ». Toute complexité ajoutée ensuite
doit se justifier par un gain mesuré par rapport à cette baseline, et ce premier run
`baseline` restera consultable dans MLflow pour cette comparaison.


## Étape 4 — Itérer : feature engineering, comparaison de modèles

**Question de réflexion :** quelles techniques des jours 1 et 2 mobiliser à ce stade, et
dans quel ordre les tester pour isoler l'effet de chacune ?


In [6]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

candidats = {
    "Régression logistique": LogisticRegression(max_iter=5000),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

scores_candidats = {}
for nom, modele in candidats.items():
    with mlflow.start_run(run_name=nom.lower().replace(" ", "_")):
        scores = cross_val_score(modele, X_train, y_train, cv=5)
        scores_candidats[nom] = scores.mean()

        mlflow.log_param("model", type(modele).__name__)
        mlflow.log_metric("cv_accuracy_mean", scores.mean())

    print(f"{nom:22s} -> score moyen CV = {scores.mean():.3f}")


View run régression_logistique at: http://127.0.0.1:5001/#/experiments/4/runs/436166df6c7e4606b4cd24b87237afca
View experiment at: http://127.0.0.1:5001/#/experiments/4
Régression logistique  -> score moyen CV = 0.803


View run random_forest at: http://127.0.0.1:5001/#/experiments/4/runs/ef543deb74ef456093b3fd18e168d517
View experiment at: http://127.0.0.1:5001/#/experiments/4
Random Forest          -> score moyen CV = 0.827


View run gradient_boosting at: http://127.0.0.1:5001/#/experiments/4/runs/e3b4ff34371644f8944eed4f69d7f5eb
View experiment at: http://127.0.0.1:5001/#/experiments/4
Gradient Boosting      -> score moyen CV = 0.831


**Ce qu'on observe** : on compare plusieurs algorithmes sur les mêmes plis de
cross-validation (question du QCM du jour 3 sur ce point précis) avant de choisir lequel
approfondir avec du feature engineering supplémentaire ou un `GridSearchCV`. Chaque
candidat teste devient un run MLflow séparé, nommé d'après le modèle — inutile de
recopier les scores dans un tableur à part.


## Étape 5 — Soumettre : suivre le score sur le leaderboard public

**Question de réflexion :** une fois le fichier `submission.csv` généré et envoyé, quel
score faut-il regarder en priorité pour décider de la suite : le score public du
leaderboard, ou le score de cross-validation obtenu localement ?


In [7]:
from sklearn.metrics import accuracy_score

meilleur_nom = max(scores_candidats, key=scores_candidats.get)
meilleur_modele = candidats[meilleur_nom]

with mlflow.start_run(run_name="submission_" + meilleur_nom.lower().replace(" ", "_")):
    meilleur_modele.fit(X_train, y_train)
    predictions = meilleur_modele.predict(X_test)
    test_accuracy = accuracy_score(y_test, predictions)

    submission = pd.DataFrame({
        "PassengerId": titanic.loc[X_test.index].index + 1,
        "Survived": predictions,
    })
    submission.to_csv("submission.csv", index=False)

    mlflow.log_param("model", type(meilleur_modele).__name__)
    mlflow.log_metric("cv_accuracy_mean", scores_candidats[meilleur_nom])
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.sklearn.log_model(meilleur_modele, name="model")
    mlflow.log_artifact("submission.csv")

print(f"Modèle retenu : {meilleur_nom}")
submission.head()


2026/09/13 17:24:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


View run submission_gradient_boosting at: http://127.0.0.1:5001/#/experiments/4/runs/339c62fabd9448deac0860a656ecea49
View experiment at: http://127.0.0.1:5001/#/experiments/4
Modèle retenu : Gradient Boosting


,PassengerId,Survived
0,710,0
1,440,0
2,841,0
3,721,1
4,40,1


**Réponse commentée** : fiez-vous d'abord à votre score de cross-validation, plus stable,
pour comparer vos propres itérations entre elles (diapositive 11 du jour 3). Le score
public du leaderboard reste un indicateur complémentaire, pas l'unique juge. Sur
http://127.0.0.1:5001, l'expérience « OFDS - Jour 3 - Compétition du jour » réunit
maintenant `baseline`, un run par candidat testé, et le run final de soumission avec le
modèle et le fichier `submission.csv` attachés — exactement la trace d'expériences
attendue par la check-list.


## Check-list avant soumission finale (diapositive 12)

Avant de considérer une soumission comme définitive, vérifiez :

1. Le pipeline de prétraitement est-il identique entre entraînement et prédiction (pas de
   fuite de données) ?
2. Le score de cross-validation est-il cohérent avec le score observé sur le leaderboard
   public ?
3. Le `random_state` est-il fixé pour garantir la reproductibilité ?
4. Le fichier de soumission respecte-t-il exactement le format attendu (colonnes, ordre,
   types) ?
5. Avez-vous gardé une trace des expériences menées (features testées, scores obtenus) ?
